# Yazıcı Kullanım Analizi — Playground

> Tek çalışan, belirli dönem ve metrik denemeleri için çalışma alanı.

Bu defter yöneticilik özetinden ayrı tutulur. Hücreler üstten alta çalıştırıldığında bağımsız biçimde çalışır; yalnızca `veriDepartment_Birlestirilmis.csv` dosyasına ihtiyaç duyar.


## Kullanım notu

İncelemek istediğiniz kişi için `personel_ozeti("kullanici_id")` çağrısını kullanın. Aşağıdaki hücrelerde örnek bir kişi sabitlenmemiştir.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
INPUT_FILE = PROJECT_DIR / "veriDepartment_Birlestirilmis.csv"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {INPUT_FILE}\n"
        "Notebook'u yaziciProje klasöründen açın veya PROJECT_DIR değerini güncelleyin."
    )

print(f"Çalışma klasörü: {PROJECT_DIR}")
print(f"Kaynak veri: {INPUT_FILE.name}")


In [ ]:
def load_and_prepare(path: Path) -> pd.DataFrame:
    """Birleştirilmiş veriyi güvenli tiplerle yükler ve analiz sütunlarını üretir."""
    df = pd.read_csv(path, sep=";", encoding="cp1254", dtype={"ID": "string"})
    df.columns = df.columns.str.strip()
    df["Tarih"] = pd.to_datetime(df["Tarih"], dayfirst=True, errors="coerce")

    numeric_columns = [
        "renkli_tek_sayfa", "renkli_cift_sayfa", "sb_tek_sayfa", "sb_cift_sayfa",
        "kopya_renkli_tek", "kopya_renkli_cift", "kopya_sb_tek", "kopya_sb_cift",
        "tarama", "toplam",
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)

    df["department"] = df["department"].fillna("").astype("string").str.strip()
    df["department_kaynak"] = df["department_kaynak"].fillna("").astype("string").str.strip()
    df["genel_renkli"] = (
        df["renkli_tek_sayfa"] + df["renkli_cift_sayfa"]
        + df["kopya_renkli_tek"] + df["kopya_renkli_cift"]
    )
    df["genel_sb"] = (
        df["sb_tek_sayfa"] + df["sb_cift_sayfa"]
        + df["kopya_sb_tek"] + df["kopya_sb_cift"]
    )
    df["dijital_islemler"] = df["tarama"]
    return df.sort_values(["Tarih", "ID"], na_position="last").reset_index(drop=True)


df = load_and_prepare(INPUT_FILE)
df.head()


## 1. Tek çalışan özeti

Fonksiyon tarihçeyi ve toplamları döndürür; veri setini değiştirmez.


In [ ]:
def personel_ozeti(df: pd.DataFrame, personel_id: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    kayıtlar = df.loc[df["ID"].astype("string").eq(str(personel_id))].copy()
    if kayıtlar.empty:
        raise ValueError(f"'{personel_id}' için kayıt bulunamadı.")

    günlük = kayıtlar[["Tarih", "department", "genel_renkli", "genel_sb", "dijital_islemler", "toplam"]].sort_values("Tarih")
    özet = günlük[["genel_renkli", "genel_sb", "dijital_islemler", "toplam"]].sum().rename("Toplam").to_frame()
    return günlük, özet


# Örnek kullanım:
# günlük, özet = personel_ozeti(df, "ornek_id")
# display(özet)
# display(günlük.head(20))


## 2. Belirli dönem ve çalışan incelemesi

Bu fonksiyon, tek bir ay ve çalışan için günlük işlemleri filtreler.


In [ ]:
def donem_personel_incelemesi(df: pd.DataFrame, personel_id: str, yil: int, ay: int) -> pd.DataFrame:
    sonuç = df.loc[
        df["ID"].astype("string").eq(str(personel_id))
        & df["Tarih"].dt.year.eq(yil)
        & df["Tarih"].dt.month.eq(ay),
        ["Tarih", "genel_renkli", "genel_sb", "dijital_islemler", "toplam"],
    ].sort_values("Tarih")
    return sonuç


# Örnek kullanım:
# donem_personel_incelemesi(df, "ornek_id", 2026, 3)


## 3. Metrik bazında sıralama

Geçerli metrikler: `toplam`, `genel_renkli`, `genel_sb`, `dijital_islemler`.


In [ ]:
def en_yuksek_kullanicilar(df: pd.DataFrame, metrik: str = "toplam", adet: int = 10) -> pd.DataFrame:
    geçerli_metrikler = {"toplam", "genel_renkli", "genel_sb", "dijital_islemler"}
    if metrik not in geçerli_metrikler:
        raise ValueError(f"Metrik şu değerlerden biri olmalı: {sorted(geçerli_metrikler)}")
    return (
        df.groupby(["ID", "isim_soyisim"], dropna=False)[metrik]
          .sum()
          .sort_values(ascending=False)
          .head(adet)
          .rename(metrik)
          .to_frame()
    )


# Örnek kullanım:
# en_yuksek_kullanicilar(df, metrik="genel_renkli", adet=15)


## 4. Veri düzenleme ve dışa aktarma

Temiz analiz tablosu ayrı oluşturulur. Bu fonksiyon açıkça çağrılmadıkça dosya yazmaz.


In [ ]:
def temiz_analiz_verisi(df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "Tarih", "ID", "isim_soyisim", "department", "department_kaynak",
        "genel_renkli", "genel_sb", "dijital_islemler", "toplam",
    ]
    return df.loc[:, columns].copy()


def temiz_veriyi_kaydet(df: pd.DataFrame, output_name: str = "analiz_verisi_temiz.csv") -> Path:
    output_path = PROJECT_DIR / output_name
    temiz_analiz_verisi(df).to_csv(output_path, sep=";", index=False, encoding="utf-8-sig")
    return output_path


# Örnek kullanım:
# temiz_veriyi_kaydet(df)


## 5. Ham günlük raporları birleştirme

Ham raporlar yeniden geldiğinde kullanılacak birleştirme fonksiyonu burada tutulur.


In [ ]:
def load_raw_reports(reports_dir: Path) -> pd.DataFrame:
    """Ham günlük yazıcı raporlarını tek tabloya birleştirir.

    Bu fonksiyon yalnızca ham raporlar yeniden işlenecekse çağrılmalıdır.
    """
    reports = sorted(reports_dir.glob("*.csv"))
    if not reports:
        raise FileNotFoundError(f"CSV raporu bulunamadı: {reports_dir}")

    frames = []
    for report in reports:
        lines = report.read_text(encoding="utf-8-sig").splitlines()
        header_row = next(
            index for index, line in enumerate(lines)
            if "Etkin Alan" in line and "Kullanıcı kimliği" in line
        )
        frame = pd.read_csv(report, skiprows=header_row, encoding="utf-8-sig")
        frame["Tarih"] = pd.to_datetime(report.stem[-10:], format="%Y-%m-%d", errors="raise")
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True).sort_values("Tarih").reset_index(drop=True)
    return combined


# Örnek kullanım (gerektiğinde yorum işaretini kaldırın):
# ham_veri = load_raw_reports(PROJECT_DIR / "reports")
